# 3 控制流

Graph 与 Chain 最大的区别：执行路径不再固定。本章共 6 节，层层递进：

1. **执行模型**（第 1 节）：super-step——LangGraph 的批处理执行循环，是理解后面所有控制流行为（并行快照、汇合、防死循环）的钥匙
2. **分支**（第 2 节）：一个节点的下游可以**并行**（2.1 静态分支）、可以**按条件选择**（2.2 条件分支）、可以在**运行时动态生成**（2.3 Send 动态分支），2.4 补充并行汇合用的 Defer 节点
3. **循环**（第 3 节）：边可以指回上游节点，反复执行直到满足退出条件——ReAct 等 agent 的核心
4. **Command**（第 4 节）：一个 Node 同时完成「更新 State」和「指定下一跳」，agent 手写循环的标准写法
5. **调试**（第 5 节）：用 `stream` 逐步观察执行路径
6. **小结**（第 6 节）：控制流选型决策表 + 实测常见坑清单

本章案例用到第 2 章的 `MessagesState` / `add_messages`，建议先学完第 2 章再读本章。

> **版本校验（langgraph 1.2.x）**：路由函数的返回值会被当作 `path_map` 的 key，因此不能返回 `Command`；实测会抛 `KeyError`（带 `update` 时可能先抛 `TypeError`），Command 只能由 Node 返回。
>
> `graph.stream(...)` 默认遇到 Node 异常会直接向调用方抛出，不会自动产生 `Error` chunk。调试时请在消费 stream 的代码外层使用 `try/except`，并可切换 `stream_mode="debug"` 查看调度事件。

# 1. 执行模型：super-step

第 1 章提过：并行会让「调度」变复杂，本章展开。先建立一个心智模型——**super-step（超步）**。

LangGraph 的执行是一个**批处理循环**（源自 Pregel 模型）：

```
初始化 State
┌─> 调度：把「当前可执行的节点」作为一批任务
│   执行：这一批节点并行跑
│   合并：每个节点的返回值合并进 State
└── 还有待执行节点？——是：进入下一个 super-step
    否：结束，返回最终 State
```

三条关键规则：

| 规则 | 含义 |
|---|---|
| **同批同快照** | 同一个 super-step 内的所有节点，读到的都是**本 super-step 开始前**的 State 快照，互相看不到对方的更新 |
| **更新即时可见** | 一个 super-step 结束、State 合并后，下一个 super-step 的节点能看到全部更新 |
| **单 key 单写** | 同一个 super-step 内，多个节点写**同一个无 reducer 的 key** 会直接报错——因为无法合并冲突 |

「同一批并行调度的节点执行完毕 + 它们的更新合并进 State」就是一个 super-step。后面所有控制流行为都能用这三条规则解释：

- 2.1 的静态分支：多个下游在**同一个** super-step 并行执行
- 2.4 的 Defer：把节点推迟到「没有其它非 defer 工作」的 super-step
- 第 3 节的 recursion_limit：按 **super-step 计数**防死循环

下面用两个纯 Python 案例验证这三条规则。

## 1.1 案例：同一 super-step 内节点读相同快照

p1、p2 由 START 同时触发 → 同一个 super-step。p2 虽然把 `count` 加 10，但 p1 读到的仍是**快照里的旧值**——两个节点都读到 1。

In [ ]:
# 案例：并行节点读相同快照（纯 Python，不依赖 LLM）
# p1 和 p2 在同一个 super-step 执行，都读到 count=1（super-step 开始前的快照值）

from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class SnapState(TypedDict):
    count: int
    seen: Annotated[list[int], add]   # 记录每个节点实际读到的 count


def p1(state: SnapState) -> dict:
    return {"seen": [state["count"]]}   # 读到快照值


def p2(state: SnapState) -> dict:
    # 即使 p2 把 count 加 10，p1 也看不到（同批同快照）
    return {"count": state["count"] + 10, "seen": [state["count"]]}


builder = StateGraph(state_schema=SnapState)
builder.add_node("p1", p1)
builder.add_node("p2", p2)
builder.add_edge(START, "p1")
builder.add_edge(START, "p2")
builder.add_edge("p1", END)
builder.add_edge("p2", END)

print(builder.compile().invoke({"count": 1, "seen": []}))
# {'count': 11, 'seen': [1, 1]}
# ← p1、p2 都读到旧值 1；count 最终值 = 快照 1 经 p2 更新后的 11

## 1.2 案例：无 reducer 并行写同一 key → 报错

w1、w2 在同一个 super-step 内都写 `v`（无 reducer）：LangGraph 无法决定保留哪个，直接抛 `InvalidUpdateError`。

**修复方式就是第 2 章的 reducer**：`Annotated[str, operator.add]` 拼接、`Annotated[list, operator.add]` 合并列表，或自定义 reducer（比如取最新）。并行写同一 key 是并行分支里**最常见的报错**（`Can receive only one value per step`），排查口诀：**并行写同一 key → 要么只让一个节点写，要么加 reducer**。

In [ ]:
# 案例：无 reducer 并行写同一 key → InvalidUpdateError

from typing import TypedDict

from langgraph.errors import InvalidUpdateError
from langgraph.graph import END, START, StateGraph


class ConflictState(TypedDict):
    v: str   # 注意：没有 reducer


def w1(state: ConflictState) -> dict:
    return {"v": "来自 w1"}


def w2(state: ConflictState) -> dict:
    return {"v": "来自 w2"}


builder = StateGraph(state_schema=ConflictState)
builder.add_node("w1", w1)
builder.add_node("w2", w2)
builder.add_edge(START, "w1")
builder.add_edge(START, "w2")
builder.add_edge("w1", END)
builder.add_edge("w2", END)

try:
    builder.compile().invoke({"v": "init"})
except InvalidUpdateError as e:
    print(f"InvalidUpdateError: {e}")
# InvalidUpdateError: At key 'v': Can receive only one value per step. Use an Annotated key to handle multiple values.

# 2. 分支结构

三种分支回答同一个问题：**一个节点执行完后，下游是谁、有几个？** 区别在于答案在什么时候确定：

| 类型 | 下游候选 | 决定时机 | 运行时行为 | 代码 |
|---|---|---|---|---|
| 静态分支（2.1） | 编译期画死的固定集合 | 编译期 | 所有下游**全部**并行执行（fan-out） | `add_edge` |
| 条件分支（2.2） | 编译期给定候选集合 | 编译期定候选，运行时选择 | 路由函数按 State 从候选中选 1 个或多个 | `add_conditional_edges` |
| 动态分支（2.3） | 运行时才生成 | 完全运行时 | Node 返回 Send，动态创建 N 个并行任务 | `Send` + `Command` |

一句话：**静态是「固定并行」，条件是「固定候选、运行时选择」，动态是「候选本身由运行时数据决定」**。三者互补，可以组合使用（比如 Send 发出的 worker 实例内部再走条件分支）。

> 结合第 1 节：静态/动态分支发出的多个下游，都在**同一个 super-step** 并行执行、读同一份快照（1.1）——并行写同一 key 要 reducer（1.2），需要汇合结果时用 2.4 的 Defer。

## 2.1 静态分支：并行 fan-out

+ 定义：从源节点出发的**多条边在图编译阶段就全部写好**，运行时这些边**全部生效**——一个节点同时触发它的所有下游（fan-out，扇出）
+ 特点：
    - 下游节点集合在编译时确定，运行时**全部**执行，不存在「选哪条」
    - 适合并行处理多个独立任务（比如同时生成笑话和诗）
    - 想根据 State 选下游 → 2.2 的条件分支；想运行时才决定有多少个并行任务 → 2.3 的 Send
+ 核心判断：编译期画好 N 条边，运行时 N 个下游都跑 -> 静态分支

**注意**：两个并行节点写同一个 key（本例的 `messages`）→ 必须带 reducer（`add_messages` 按 id 去重合并），否则触发 1.2 的 `InvalidUpdateError`。

In [ ]:
# 示例：两个 LLM 节点并行执行（标准格式）
# 标准 LLM 节点格式：读完整历史 state["messages"]，返回 {"messages": [response]}
# 两个并行节点写同一个 key（messages）→ add_messages 按 id 去重合并，两条回复都保留

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"           # 用 name 标记角色，便于区分并行节点的输出
    return {"messages": [response]}  # 标准格式：返回消息增量，走 add_messages 合并


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
builder.add_edge(START, "joke_node")
builder.add_edge(START, "poem_node")
builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)

graph = builder.compile()
result = graph.invoke({
    "topic": "猫",
    "messages": [HumanMessage(content="请围绕这个主题展示你的才华")],
})
rprint(result)

# 显示图结构：两条静态边从 START 并行出发
display(graph)
# 输出示例（截断）:
# messages = [HumanMessage("请围绕这个主题展示你的才华"),
#             AIMessage(name="joke", content="（一个关于猫的笑话……）"),
#             AIMessage(name="poem", content="（一首关于猫的诗……）")]
# 两条并行节点的回复都被 add_messages 保留（并行写同一 key 必须有 reducer，见 1.2）


## 2.2 条件分支

与静态分支不同，条件分支的**下游节点由运行时函数决定**：节点执行完后，LangGraph 调用一个**路由函数**读取 State，根据返回值从映射表里选出下一个节点。

+ 定义：路由函数 `(state) -> str`，返回值对应映射表里的 key，决定下一跳
+ 特点：
    - 下游候选集合在编译时通过映射表确定
    - **运行时**才决定走哪条边（路由函数读 State 判断）
    - 一个来源可以同时有多条条件边（多目标分支）
+ 与静态分支对比：
    - 静态分支（并行分支）：所有下游**同时**执行，运行前就确定
    - 条件分支：候选下游**二选一**，必须等路由函数跑完才知道去向
+ 核心判断：路由函数的返回值决定下游 -> 条件分支
+ 代码：`add_conditional_edges(源节点, 路由函数, {key: 目标节点})`

```python
# 三个参数：从哪个节点出发、路由函数、返回值 → 目标节点 的映射表
builder.add_conditional_edges("a", route, {"x": "b", "y": "c"})
```

**路由函数参数说明**：

- 签名：`def route(state) -> str`，与 Node 一样，第一个参数是**完整 State 快照**，按 key 读取任何需要的字段（全局 State、或私有 schema 中声明的同名 key），据此判断走向
- 返回值：必须是映射表的 **key**（字符串）；返回值不在映射表里会直接抛 `KeyError`（编译期不检查，所有分支路径都要测到）
- 变体：
    - 返回 `list[str]` → 同时走向**多个**目标（条件多分支，同一 super-step 并行）
    - 返回 `Send` 或 `Send` 列表 → 动态创建并行任务（见 2.3）
    - **不能返回 `Command`**：`Command` 只能由 **Node** 返回（第 4 节）。路由函数返回 `Command` 不会报错，但**跳转和更新都会被静默忽略**，是隐蔽的坑
- 调用时机：源节点执行完后、下一个 super-step 调度时调用；同一个路由函数每次执行都可能返回不同结果

**`path_map` 的三种写法**：

| 写法 | 行为 |
|---|---|
| `path_map={"joke": "joke_node", ...}`（dict） | 返回值经映射表翻译成节点名（返回值和节点名可以不一致） |
| `path_map=["joke_node", "poem_node"]`（list） | 等价于 `{节点名: 节点名}` 映射 |
| 省略 `path_map` | **返回值本身就被当作节点名**（或 `Send` 原样投递）；没有 `Literal` 返回注解时，图可视化会假设可以跳到任何节点 |

条件分支是 LangGraph 最重要的控制流结构之一：本章第 3 节的循环（写诗 → 检查 → 重写）和第 2 章 3.1.2 的 plan → route → plan/answer 都是它实现的；ReAct 的「思考 → 行动 → 观察」循环同理。

下面给出与并行分支相同主题（主题创作）的条件分支案例。

In [ ]:
# 示例：条件分支 —— 同一组节点，运行时按 kind 选择走哪条边
# 与并行分支对比：并行是两条都跑，条件是根据 State 二选一

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str
    kind: str   # "joke" | "poem"，路由函数根据它选择下游


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"
    return {"messages": [response]}


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


# 路由函数：读 State，返回映射表的 key
def route(state: ChatState) -> str:
    return state["kind"]


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
# 条件边：直接从 START 出发，运行时根据 route 的返回值选择下游
# path_map 显式命名：路由函数返回值 → 目标节点 的映射表
builder.add_conditional_edges(START, route, path_map={"joke": "joke_node", "poem": "poem_node"})
builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)

graph = builder.compile()

# 同一组节点，两次 invoke 走不同路径
result = graph.invoke({"topic": "猫", "kind": "joke", "messages": [HumanMessage(content="请开始表演")]})
rprint([(m.name, m.content[:30]) for m in result["messages"] if m.name])

result = graph.invoke({"topic": "猫", "kind": "poem", "messages": [HumanMessage(content="请开始表演")]})
rprint([(m.name, m.content[:30]) for m in result["messages"] if m.name])

# 显示图结构：条件边从 START 出发，二选一
display(graph)
# 输出示例（截断）:
# [('joke', '好的，灯光就位，观众入座，现在开始表演——…')]     ← kind="joke" 那次
# [('poem', '## 《猫》…')]                                      ← kind="poem" 那次


下面把条件分支扩展到**三个目标**：在主题创作中新增「国歌」节点，路由函数根据 `kind` 三选一。

In [ ]:
# 示例：条件分支三选一 —— 路由到国歌节点
# 与二选一相比只多了一个候选节点和一条映射项，路由函数不变

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str
    kind: str   # "joke" | "poem" | "anthem"


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"
    return {"messages": [response]}


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


# 新增的国歌节点：与 joke / poem 完全同构，只是提示词和 name 不同
def anthem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」创作一首国歌")]
    )
    response.name = "anthem"
    return {"messages": [response]}


def route(state: ChatState) -> str:
    return state["kind"]


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
builder.add_node("anthem_node", anthem_node)
# 三选一：路由函数返回值决定走 joke / poem / anthem 中的哪条
builder.add_conditional_edges(START, route, path_map={
    "joke": "joke_node",
    "poem": "poem_node",
    "anthem": "anthem_node",
})
builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)
builder.add_edge("anthem_node", END)

graph = builder.compile()

# 同一组节点，三种 kind 各走一条路径
for kind in ["joke", "poem", "anthem"]:
    result = graph.invoke({"topic": "猫", "kind": kind, "messages": [HumanMessage(content="请开始表演")]})
    rprint([(m.name, m.content[:30]) for m in result["messages"] if m.name])

# 显示图结构：条件边从 START 出发，三选一
display(graph)
# 输出示例（截断）: 三次 invoke 分别输出
# [('joke', '好的，演出开始。…')] / [('poem', '## 《猫》…')] / [('anthem', '…')]


路由函数不仅支持**三选一**（返回单个 key），还可以返回 `list[str]` **同时走向多个目标**——三选一 / 三选二 / 三选三，都由同一个映射表决定：

| 路由返回值 | 效果 | 对应写法 |
|---|---|---|
| `"joke"` | 3 选 1：只走一个节点 | 返回单个字符串 |
| `["joke", "poem"]` | 3 选 2：两个节点**同一 super-step 并行**执行 | 返回字符串列表 |
| `["joke", "poem", "anthem"]` | 3 选 3：三个节点全部执行（等价于并行分支的 `add_edge` 写法） | 返回完整列表 |

注意：多目标并行执行时，`messages` 由 `add_messages` 按**完成顺序**合并——谁先返回谁在前，顺序不保证（见下方输出）。

In [ ]:
# 示例：3 选 1 / 3 选 2 / 3 选 3 —— 路由函数返回 list[str] 同时走向多个目标
# 与上一个案例的区别：kinds 是列表，route_multi 原样返回它

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str
    kinds: list[str]   # 需要哪几个就放哪几个：["joke"] / ["joke","poem"] / 全部


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"
    return {"messages": [response]}


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


def anthem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」创作一首国歌")]
    )
    response.name = "anthem"
    return {"messages": [response]}


# 路由函数返回 list[str]：列表里的每个 key 都对应一个目标节点
def route_multi(state: ChatState) -> list[str]:
    return state["kinds"]


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
builder.add_node("anthem_node", anthem_node)
# 同一个映射表：单个字符串走一条，列表同时走多条
builder.add_conditional_edges(START, route_multi, path_map={
    "joke": "joke_node",
    "poem": "poem_node",
    "anthem": "anthem_node",
})
builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)
builder.add_edge("anthem_node", END)

graph = builder.compile()

# 3 选 1 / 3 选 2 / 3 选 3
for kinds in (["joke"], ["joke", "poem"], ["joke", "poem", "anthem"]):
    result = graph.invoke({"topic": "猫", "kinds": kinds, "messages": [HumanMessage(content="请开始表演")]})
    names = [m.name for m in result["messages"] if m.name]
    print(f"请求 {kinds} -> 实际执行: {names}")

# 显示图结构：条件边从 START 出发，三路候选，运行时可同时走多条
display(graph)
# 输出示例:
# 请求 ["joke"] -> 实际执行: ["joke"]
# 请求 ["joke", "poem"] -> 实际执行: ["joke", "poem"]
# 请求 ["joke", "poem", "anthem"] -> 实际执行: ["anthem", "joke", "poem"]   ← 顺序不保证


## 2.2.1 陷阱：普通边和条件边可以共存，且**两条都走**

一个节点可以**同时**有普通边和条件边。注意语义：普通边是「一定走」，条件边是「路由说了算」，两者是**叠加**而不是二选一——下面的 mid 既走普通边到 b，又走条件边到 c，b、c 在同一个 super-step 并行执行。

| 需求 | 正确写法 |
|---|---|
| 下游固定多个，全部执行 | 只用普通边 `add_edge` |
| 下游按 State 选一个/多个 | 只用条件边 `add_conditional_edges` |
| 「固定走 A，另外视情况走 B」 | 普通边 + 条件边共存（叠加语义） |

> 同样的叠加语义也发生在第 4 节：Node 返回 `Command(goto=...)` 时，该节点的静态边**仍然执行**。

In [ ]:
# 案例：普通边 + 条件边共存 —— 两条都走（叠加语义）
# mid → b 是普通边（一定走）；mid → c 是条件边（route 返回 "c"）
# 实际执行：b、c 都跑，在同一个 super-step 并行

from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class BothState(TypedDict):
    log: Annotated[list[str], add]


def mid(state: BothState) -> dict:
    return {}


def target_b(state: BothState) -> dict:
    return {"log": ["via 普通边"]}


def target_c(state: BothState) -> dict:
    return {"log": ["via 条件边"]}


def route(state: BothState) -> str:
    return "c"


builder = StateGraph(state_schema=BothState)
builder.add_node("mid", mid)
builder.add_node("b", target_b)
builder.add_node("c", target_c)
builder.add_edge(START, "mid")
builder.add_edge("mid", "b")                             # 普通边
builder.add_conditional_edges("mid", route, {"c": "c"})  # 条件边
builder.add_edge("b", END)
builder.add_edge("c", END)

print(builder.compile().invoke({"log": []}))
# {'log': ['via 普通边', 'via 条件边']}   ← 两条都走（输出顺序不保证）

## 2.3 动态分支（Send API）

前两种分支的下游集合都在**编译期**确定。但有些场景编译期根本不知道下游有几个：比如「把任务拆成 N 个子任务并行处理」，N 是运行时才知道的。

`Send` 解决这个问题：Node 返回 `Command(goto=[Send(目标节点, 数据), ...])`，在**运行时动态创建并行任务**（map-reduce 的 map 部分）。每个 Send 相当于「向目标节点再投递一份输入」：

- `Send(node, payload)`：向 `node` 投递 `payload`，`payload` 会与当前 State 合并，成为该 node 的一次输入
- 返回几个 `Send`，目标节点就**并行**执行几次
- `Send` 从 `langgraph.types` 导入（与 `Command` 同源）

**两种等价写法**（运行结果相同，怎么选看逻辑放哪更顺）：

| 写法 | 说明 |
|---|---|
| **Node 返回** `Command(goto=[Send(...), ...])` | 推荐：Node 一边更新 State 一边动态分发（第 4 节） |
| **路由函数返回** `Send` 列表 | `add_conditional_edges("planner", route)`，route 返回 `[Send("worker", {...}), ...]` |

下面演示：planner 运行时把 3 个任务分发给 worker，worker 并行处理，结果用 reducer 汇总（纯 Python，不依赖 LLM）。

In [1]:
# 示例：Send 动态分支 —— planner 运行时把 N 个任务分发给 worker 并行处理
# 与静态分支的区别：并行任务的数量（3 个 worker 实例）是运行时由 planner 决定的

from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send


class SendState(TypedDict):
    tasks: list[str]                    # 待处理任务
    results: Annotated[list[str], add]  # worker 结果，reducer 汇总


def planner(state: SendState) -> Command:
    # 每个任务向 worker 投递一份输入；worker 会并行执行 3 次
    return Command(goto=[Send("worker", {"task": t}) for t in state["tasks"]])


def worker(state) -> dict:
    # Send 的 payload 已合并进本次输入的 State，直接读 task
    return {"results": [f"处理完成: {state['task']}"]}


builder = StateGraph(state_schema=SendState)
builder.add_node("planner", planner)
builder.add_node("worker", worker)
builder.add_edge(START, "planner")
builder.add_edge("planner", END)   # planner 发完任务就结束自己的路径
builder.add_edge("worker", END)    # 每个 worker 实例结束后各自走向 END

print(builder.compile().invoke({"tasks": ["a", "b", "c"], "results": []}))
# {'tasks': ['a', 'b', 'c'], 'results': ['处理完成: a', '处理完成: b', '处理完成: c']}

{'tasks': ['a', 'b', 'c'], 'results': ['处理完成: a', '处理完成: b', '处理完成: c']}


In [3]:
# 案例：路由函数返回 Send 列表 —— 与「Node 返回 Command(goto=[Send])」等价
# 区别：planner 只更新 State；分发逻辑集中在路由函数里

from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class FanoutState(TypedDict):
    tasks: list[str]
    results: Annotated[list[str], add]


def planner(state: FanoutState) -> dict:
    return {}   # Node 只做自己的事，不决定分发


def route(state: FanoutState) -> list[Send]:
    # 路由函数按 State 动态生成 Send 列表：有几个任务就分发几个
    return [Send("worker", {"task": t}) for t in state["tasks"]]


def worker(state) -> dict:
    return {"results": [f"处理完成: {state['task']}"]}


builder = StateGraph(state_schema=FanoutState)
builder.add_node("planner", planner)
builder.add_node("worker", worker)
builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", route)   # 省略 path_map：Send 原样投递
builder.add_edge("worker", END)

print(builder.compile().invoke({"tasks": ["x", "y"], "results": []}))
# {'tasks': ['x', 'y'], 'results': ['处理完成: x', '处理完成: y']}

{'tasks': ['x', 'y'], 'results': ['处理完成: x', '处理完成: y']}


## 2.4 Defer 节点：并行分支的汇合点

`add_node(..., defer=True)`：让节点**推迟到调度器没有其它非 defer 工作可做时**再执行（`defer` 是 `StateNodeSpec` 上的参数，从 v0.6 起支持）。

- 适用场景：并行分支的**汇合点**（map-reduce、共识、多 agent 汇总）——聚合节点要等所有分支的结果都落地后再跑
- 语义：
  - defer 只是"推迟到无其它非 defer 工作"，**不是**"等所有祖先节点"；节点仍可能执行多次
  - 想让某个节点只在指定前提满足时跑，应显式用条件边 / `Command(goto=...)` 控制
- 对比（下面案例演示）：

| 写法 | 行为 |
|---|---|
| `add_node("aggregate", aggregate)` | 与并行分支**同一 super-step** 执行，读到的 State 里**没有**分支结果 |
| `add_node("aggregate", aggregate, defer=True)` | 分支先跑完（super-step 1），聚合节点在后续 super-step 执行，能看到**全部**结果 |

> 用第 1 节的规则解释：不 defer 时 aggregate 与 joke/poem **同批**，读的是 super-step 开始前的**快照**（1.1 同批同快照）；defer 把它挪到之后的 super-step，自然能看到合并后的全部结果。

下面用主题创作演示：joke / poem 并行，聚合节点用 defer 等到两条回复都落地后再统计。

In [ ]:
# 示例：Defer 节点 —— 聚合节点推迟到并行分支全部完成后再执行
# 对比：不加 defer 时，aggregate 与 joke/poem 同一 super-step 并行，读不到分支结果

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str
    summary: str


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"
    return {"messages": [response]}


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


# 聚合节点：defer=True → 推迟到没有其它非 defer 工作可做时才执行
def aggregate(state: ChatState) -> dict:
    names = [m.name for m in state["messages"] if m.name]
    print(f"[aggregate] 执行时看到的 AI 回复: {names}")
    return {"summary": f"共收到 {len(names)} 条创作: {names}"}


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
builder.add_node("aggregate", aggregate, defer=True)   # ← defer 节点
builder.add_edge(START, "joke_node")
builder.add_edge(START, "poem_node")
builder.add_edge(START, "aggregate")   # aggregate 也被 START 触发，但 defer 推迟执行
builder.add_edge("aggregate", END)

graph = builder.compile()

result = graph.invoke({"topic": "猫", "messages": [HumanMessage(content="请开始表演")]})
rprint(result["summary"])
# 输出示例: 共收到 2 条创作: ['joke', 'poem']   ← 两条并行回复都已落地

# 显示图结构：三条边从 START 出发，aggregate 是 defer 节点
display(graph)

# 3. 循环结构

循环是 LangGraph 最核心的能力：**边可以指回上游节点**，让 Graph 反复执行，直到条件边放行到 END。ReAct（思考 → 行动 → 观察 → 再思考）、生成-检查-重写都是循环。

两个关键点：

- 循环**必须**配合条件边（或第 4 节的 `Command(goto=...)`）：否则图会一直转下去。LangGraph 用 **recursion_limit** 兜底——**按 super-step 计数**（默认 25），超过就抛 `GraphRecursionError`
- 每转一圈 State 都保留：`add_messages`、计数器 reducer 让 State 随循环累积——这是循环「越跑越接近目标」的前提

In [ ]:
# 示例：计数器循环 —— 转 3 圈自动退出（纯 Python，不依赖 LLM）

from operator import add
from typing import Annotated, TypedDict

from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph


class LoopState(TypedDict):
    count: Annotated[int, add]


def inc(state: LoopState) -> dict:
    print(f"第 {state['count']} 圈")
    return {"count": 1}


def route(state: LoopState) -> str:
    return "loop" if state["count"] < 3 else END   # 满 3 圈放行到 END


builder = StateGraph(state_schema=LoopState)
builder.add_node("inc", inc)
builder.add_edge(START, "inc")
# 循环的关键：条件边的映射把 inc 指回自己；END 也放进映射表，route 返回 END 时直接结束
builder.add_conditional_edges("inc", route, {"loop": "inc", END: END})

print(builder.compile().invoke({"count": 0}))
# 第 0 圈
# 第 1 圈
# 第 2 圈
# {'count': 3}


# recursion_limit 兜底：故意让循环永不退出，验证防死循环保护（默认上限 25 步）
def always_loop(state: LoopState) -> str:
    return "loop"


builder2 = StateGraph(state_schema=LoopState)
builder2.add_node("inc", inc)
builder2.add_edge(START, "inc")
builder2.add_conditional_edges("inc", always_loop, {"loop": "inc"})

try:
    builder2.compile().invoke({"count": 0}, config={"recursion_limit": 8})
except GraphRecursionError as e:
    print("触发了防死循环保护:", str(e)[:60])

In [ ]:
# 示例：LLM 自纠错循环 —— 写诗直到包含「月亮」，不合格就重写

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class PoemState(MessagesState):
    ok: bool


def write_poem(state: PoemState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content="写一首包含「月亮」两个字的短诗")]
    )
    return {"messages": [response]}


def check(state: PoemState) -> dict:
    last = state["messages"][-1]
    ok = "月亮" in (last.content or "")
    print(f"检查: 诗中{'包含' if ok else '不含'}月亮")
    return {"ok": ok}


def route(state: PoemState) -> str:
    return "finish" if state["ok"] else "write_poem"   # 不合格 → 指回写诗节点重写


def finish(state: PoemState) -> dict:
    return {}


builder = StateGraph(state_schema=PoemState)
builder.add_node("write_poem", write_poem)
builder.add_node("check", check)
builder.add_node("finish", finish)
builder.add_edge(START, "write_poem")
builder.add_edge("write_poem", "check")
# 循环的关键：check 的条件边把 write_poem 指回自己，同时保留通往 finish 的分支
builder.add_conditional_edges("check", route, {"write_poem": "write_poem", "finish": "finish"})
builder.add_edge("finish", END)

result = builder.compile().invoke({"messages": [HumanMessage(content="开始")]})
rprint(result["messages"][-1].content)
# 输出示例（截断）:
# 检查: 诗中包含月亮      ← 第一轮就合格，直接结束
# ## 《夜航西飞》…
# （若不合格会打印「检查: 诗中不含月亮」，然后指回 write_poem 重写）


# 4. Command：跳转与更新二合一

前面 Node 都是「返回 dict 更新 State」，跳转交给边。`Command` 让 Node 在返回时**同时**写 State 和指定下一跳：

- `Command(goto=..., update=...)`：`goto` 指定下一个节点，`update` 是 State 更新（正常走 reducer 合并）
- 适用场景：agent 手写循环（模型输出 Command 直接指定下一步并携带数据）、「边跑边决定」的复杂控制流
- `Command` 从 `langgraph.types` 导入

**完整能力清单**（本节逐个演示）：

| 能力 | 写法 | 案例 |
|---|---|---|
| 指定下一跳 + 更新 | `Command(goto="b", update={...})` | 4.1 |
| `goto` 多目标并行 | `Command(goto=["b", "c"])` | 4.2 |
| 直接结束 | `Command(goto=END)` | 4.3（猜数字 agent） |
| 动态分发 | `Command(goto=[Send(...), ...])` | 2.3 已演示 |

**三条规则**（都容易踩坑）：

1. **返回类型注解**：返回 `Command` 的 Node 建议标注 `-> Command[Literal["b", "c", "__end__"]]`（`END` 写作 `"__end__"`）。注解只影响**图可视化和静态检查**、不影响运行，但漏掉会让 `display(graph)` 画不出这条边。也可以在 `add_node(..., destinations=...)` 里声明（同样只用于渲染）
2. **静态边仍然执行**：`Command(goto=...)` 是「额外跳转」而非「替代出边」——若同时 `add_edge("a", "b")` 且 `goto="c"`，**b 和 c 都执行**；若静态边与 goto 指向**同一个**节点 b，则 b 只执行**一次**（去重）。每个节点要么用 Command 要么用静态边，不要混用
3. **Command 只能由 Node 返回**：路由函数返回 `Command` 会被**静默忽略**（跳转和更新都丢失，2.2 已提）；需要「路由 + 更新」时把逻辑移进 Node

> 等价关系：`return Command(goto="b", update={...})` ≈ `return {...}` + 条件边路由到 b。Command 把两件事合成一个返回值，让 Node 自己决定走向。

## 4.1 基础用法：goto + update

节点 a 没有**任何出边**：它的走向完全由返回的 `Command.goto` 决定——count ≥ 2 去 b，否则去 c，同时把 `log` 更新写进 State。

In [ ]:
# 案例：Command —— 一个 Node 同时更新 State 并指定下一跳（纯 Python）

from operator import add
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command


class CmdState(TypedDict):
    count: Annotated[int, add]
    log: str


# 注解 Command[Literal[...]] 只影响图可视化，不影响运行；建议写上
def a(state: CmdState) -> Command[Literal["b", "c"]]:
    # 不靠边决定走向：直接返回 Command，边更新 State 边指定下一跳
    if state["count"] >= 2:
        return Command(goto="b", update={"log": "a -> b", "count": 1})
    return Command(goto="c", update={"log": "a -> c", "count": 1})


def b(state: CmdState) -> dict:
    return {"log": state["log"] + " -> 结束于 b"}


def c(state: CmdState) -> dict:
    return {"log": state["log"] + " -> 结束于 c"}


builder = StateGraph(state_schema=CmdState)
builder.add_node("a", a)
builder.add_node("b", b)
builder.add_node("c", c)
builder.add_edge(START, "a")
# a 没有出边：它的走向完全由返回的 Command.goto 决定
builder.add_edge("b", END)
builder.add_edge("c", END)

print(builder.compile().invoke({"count": 2, "log": ""}))
# {'count': 3, 'log': 'a -> b -> 结束于 b'}

## 4.2 多目标 goto 与静态边共存

`goto` 传**列表** → 多个目标在同一 super-step 并行执行（效果等同路由函数返回 `list[str]`）。同时验证规则 2 的去重行为：a 返回 `Command(goto=["b", "c"])`，但 a→b 还有一条静态边——观察 b 是执行**一次**还是两次。

In [ ]:
# 案例：goto 多目标 + 静态边共存 —— 叠加且去重
# a 返回 Command(goto=["b", "c"])，同时 a→b 还有静态边：
#   - goto 指向的 b、c 都执行
#   - 静态边也生效，但与 goto 重叠的 b 只执行一次（去重）
# 若静态边指向别的节点（如 a→d），则 b、c、d 都会执行

from operator import add
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command


class MultiState(TypedDict):
    log: Annotated[list[str], add]


def a(state: MultiState) -> Command[Literal["b", "c"]]:
    return Command(goto=["b", "c"], update={"log": ["a"]})


def b(state: MultiState) -> dict:
    return {"log": ["b 执行"]}


def c(state: MultiState) -> dict:
    return {"log": ["c 执行"]}


builder = StateGraph(state_schema=MultiState)
builder.add_node("a", a)
builder.add_node("b", b)
builder.add_node("c", c)
builder.add_edge(START, "a")
builder.add_edge("a", "b")   # 静态边与 goto 中的 b 重叠
builder.add_edge("b", END)
builder.add_edge("c", END)

print(builder.compile().invoke({"log": []}))
# {'log': ['a', 'b 执行', 'c 执行']}   ← b 只执行一次（去重），不是两次

## 4.3 实战：猜数字 agent——LLM 手写循环

一个最小的「模型驱动控制流」案例：agent 节点让 LLM 猜 1~10 的数字，猜错带着反馈（大了/小了）继续猜，猜对或超过轮数上限直接 `Command(goto=END)`。整个循环**没有任何节点间静态边**——走向完全由模型输出决定，这就是 Command 的典型用法（ReAct、多 agent supervisor 同理）。

要点：

- `model.with_structured_output(Guess)`：让模型输出结构化的猜测（Pydantic 类约束字段）
- `-> Command[Literal["agent", "__end__"]]`：声明可能的目的地（含 END）
- `tries` 用 reducer 累计轮数，作为防死循环的**第二道保险**（第一道是 recursion_limit）

In [ ]:
# 案例：猜数字 agent —— LLM 决定继续猜还是结束（Command 手写循环）
# 没有任何节点间静态边：agent 的走向完全由返回的 Command 决定

from operator import add
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from pydantic import BaseModel

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class Guess(BaseModel):
    number: int
    reason: str


guesser = model.with_structured_output(Guess)


class GuessState(TypedDict):
    target: int
    tries: Annotated[int, add]           # 轮数（reducer 累计，防死循环第二道保险）
    history: Annotated[list[str], add]   # 猜测历史 + 反馈，喂给下一轮


def agent(state: GuessState) -> Command[Literal["agent", "__end__"]]:
    history = "; ".join(state["history"]) or "还没有猜测记录"
    prompt = (
        f"我们在玩猜数字游戏，目标是一个 1~10 的整数。"
        f"之前的猜测记录：{history}。"
        f"请综合记录给出你的下一个猜测，并说明理由。"
    )
    g = guesser.invoke([HumanMessage(content=prompt)])
    if g.number == state["target"]:
        feedback = "猜对了！"
    elif g.number > state["target"]:
        feedback = "猜大了"
    else:
        feedback = "猜小了"
    entry = f"猜 {g.number}（{g.reason}）→ {feedback}"
    print(f"第 {state['tries'] + 1} 轮: {entry}")
    updates = {"tries": 1, "history": [entry]}
    if g.number == state["target"] or state["tries"] >= 7:
        return Command(goto=END, update=updates)   # 结束：猜对或达到轮数上限
    return Command(goto="agent", update=updates)   # 继续：带着反馈再猜


builder = StateGraph(state_schema=GuessState)
builder.add_node("agent", agent)
builder.add_edge(START, "agent")   # 只有入口边；出边由 Command 决定

result = builder.compile().invoke({"target": 7, "tries": 0, "history": []})
print(f"共 {result['tries']} 轮结束")
# 输出示例（每次运行不同）:
# 第 1 轮: 猜 5（从中间开始二分）→ 猜小了
# 第 2 轮: 猜 8（在 6~10 之间取中）→ 猜大了
# 第 3 轮: 猜 7（介于 6 和 8 之间）→ 猜对了！
# 共 3 轮结束

# 5. 调试控制流：stream

分支 + 循环 + Command 组合起来，执行路径越来越难推断。`invoke` 只给最终 State，`stream` 让你**逐步观察执行过程**：

| stream_mode | 每个 chunk 是什么 | 适用 |
|---|---|---|
| `"updates"`（最常用） | **一次节点执行**的更新：`{节点名: 该节点的返回值}` | 看**谁在执行、返回了什么**——控制流调试首选；并行节点各占一个 chunk |
| `"values"` | 一个 super-step 后的**完整 State 快照** | 看 State 如何随步骤演化（同批节点的更新会合并进同一份快照） |
| `"debug"` | 全部事件（含调度细节） | 深度排查 |

节点抛异常时，`updates` 模式里该节点的值是一个 `Error` 对象，方便定位是哪个节点挂了。

下面用「分支 + 汇合 + 循环」的小图同时演示两种模式：start 并行发出 left/right，join 汇合后视情况回环再转一圈。

In [ ]:
# 案例：stream_mode="updates" / "values" 观察执行路径
# 图结构：start 并行发出 left/right（同一 super-step）→ join 汇合 → 视情况回环

from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class StreamState(TypedDict):
    log: Annotated[list[str], add]
    round: Annotated[int, add]


def start(state: StreamState) -> dict:
    return {"log": ["start"], "round": 1}


def left(state: StreamState) -> dict:
    return {"log": ["left"]}


def right(state: StreamState) -> dict:
    return {"log": ["right"]}


def join(state: StreamState) -> dict:
    return {"log": [f"join(第{state['round']}轮)"]}


def route(state: StreamState) -> str:
    return "loop" if state["round"] < 2 else "done"   # 转 2 圈退出


builder = StateGraph(state_schema=StreamState)
builder.add_node("start", start)
builder.add_node("left", left)
builder.add_node("right", right)
builder.add_node("join", join)
builder.add_edge(START, "start")
builder.add_edge("start", "left")
builder.add_edge("start", "right")   # left/right 同一 super-step 并行
builder.add_edge("left", "join")
builder.add_edge("right", "join")
builder.add_conditional_edges("join", route, {"loop": "start", "done": END})

graph = builder.compile()

print("=== stream_mode='updates'：逐节点观察执行路径 ===")
for chunk in graph.stream({"log": [], "round": 0}):
    print(chunk)
# {'start': {'log': ['start'], 'round': 1}}
# {'left': {'log': ['left']}}
# {'right': {'log': ['right']}}
# {'join': {'log': ['join(第1轮)']}}
# {'start': {'log': ['start'], 'round': 1}}          ← 回环：第 2 圈开始
# {'left': {'log': ['left']}}
# {'right': {'log': ['right']}}
# {'join': {'log': ['join(第2轮)']}}

print("=== stream_mode='values'：每步后的完整 State 快照 ===")
for snapshot in graph.stream({"log": [], "round": 0}, stream_mode="values"):
    print(snapshot)
# {'log': [], 'round': 0}                                       ← 初始
# {'log': ['start'], 'round': 1}
# {'log': ['start', 'left', 'right'], 'round': 1}                ← left/right 已合并
# {'log': ['start', 'left', 'right', 'join(第1轮)'], 'round': 1}
# {'log': ['start', 'left', 'right', 'join(第1轮)', 'start'], 'round': 2}
# {'log': ['start', 'left', 'right', 'join(第1轮)', 'start', 'left', 'right'], 'round': 2}
# {'log': ['start', 'left', 'right', 'join(第1轮)', 'start', 'left', 'right', 'join(第2轮)'], 'round': 2}

## 5.1 无 API Key 综合练习：路由、并行、汇合与循环

前面的 LLM 案例展示了真实应用，但学习控制流不应该依赖外部服务。下面用一个**订单处理流程**把本章的核心机制串起来：

1. `classify` 根据订单类型做条件路由；
2. `prepare` 和 `notify` 通过静态边并行执行；
3. 两个节点都写 `events`，用 `operator.add` 合并；
4. `review` 汇合后检查库存，不足时回到 `restock`；
5. `restock` 设置库存后再次进入 `review`，满足条件后结束。

阅读代码时重点观察：同一 super-step 的并行节点读的是同一份快照，多个节点写同一 key 必须使用 reducer，循环必须有明确的业务退出条件。

In [ ]:
# 纯 Python 综合案例：条件路由 -> 静态并行 -> 汇合 -> 有上限循环
# 可直接复制为 .py 文件运行，不需要 API Key

from operator import add
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command


class OrderState(TypedDict):
    order_type: Literal["normal", "priority"]
    stock: int
    need: int
    retries: Annotated[int, add]
    events: Annotated[list[str], add]


def classify(state: OrderState) -> Command[Literal["normal_prepare", "priority_prepare"]]:
    # Command 同时记录事件并决定下一跳；这里也可以改成条件边实现
    target = "priority_prepare" if state["order_type"] == "priority" else "normal_prepare"
    return Command(goto=target, update={"events": [f"分类:{state['order_type']}"]})


def normal_prepare(state: OrderState) -> dict:
    return {"events": ["普通订单准备"]}


def priority_prepare(state: OrderState) -> dict:
    return {"events": ["加急订单准备"]}


def package(state: OrderState) -> dict:
    # 与 notify 在同一 super-step 并行执行
    return {"events": [f"打包({state['need']}件)"]}


def notify(state: OrderState) -> dict:
    return {"events": ["通知仓库"]}


def review(state: OrderState) -> dict:
    enough = state["stock"] >= state["need"]
    status = "库存充足" if enough else "库存不足"
    return {"events": [f"复核:{status}"]}


def route_after_review(state: OrderState) -> str:
    # 业务退出条件：库存足够；保护条件：最多补货 2 次
    if state["stock"] >= state["need"]:
        return "done"
    if state["retries"] >= 2:
        return "failed"
    return "restock"


def restock(state: OrderState) -> dict:
    return {"stock": 2, "retries": 1, "events": ["补货: +2"]}


def finish(state: OrderState) -> dict:
    return {"events": ["订单完成"]}


def fail(state: OrderState) -> dict:
    return {"events": ["订单失败: 超过补货上限"]}


builder = StateGraph(state_schema=OrderState)
for name, node in {
    "classify": classify,
    "normal_prepare": normal_prepare,
    "priority_prepare": priority_prepare,
    "package": package,
    "notify": notify,
    "review": review,
    "restock": restock,
    "finish": finish,
    "fail": fail,
}.items():
    builder.add_node(name, node)

builder.add_edge(START, "classify")
# Command 选出一个准备节点；两个准备节点共享同一段后续流程
builder.add_edge("normal_prepare", "package")
builder.add_edge("priority_prepare", "package")
# package 和 notify 是固定并行分支，下一节点 review 会在两者完成后执行
builder.add_edge("package", "review")
builder.add_edge("package", "notify")
builder.add_edge("notify", "review")
builder.add_conditional_edges("review", route_after_review, {
    "restock": "restock",
    "done": "finish",
    "failed": "fail",
})
builder.add_edge("restock", "review")
builder.add_edge("finish", END)
builder.add_edge("fail", END)

graph = builder.compile()
result = graph.invoke({
    "order_type": "priority",
    "stock": 0,
    "need": 2,
    "retries": 0,
    "events": [],
})
print(result)
# 关键输出（events 的并行部分顺序可能不同）：
# events 包含 分类:priority、加急订单准备、打包(2件)、通知仓库、
#       复核:库存不足、补货: +2、复核:库存充足、订单完成
# retries == 1，说明循环只执行了一次补货就满足退出条件

## 5.2 练习：把条件分支从二选一扩展到多目标

条件路由函数既可以返回一个 key，也可以返回 key 列表。下面这个最小案例不调用 LLM，专门验证三种输入：只运行一个节点、并行运行两个节点、运行全部节点。注意 `events` 使用 reducer，因此并行写入不会触发 `InvalidUpdateError`；输出列表的顺序不应作为业务逻辑依据。

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class FanoutState(TypedDict):
    selected: list[str]
    events: Annotated[list[str], add]


def route(state: FanoutState) -> list[str]:
    # selected 来自调用方，因此候选节点必须通过 path_map 白名单限制
    return state["selected"]


def make_node(name: str):
    def node(state: FanoutState) -> dict:
        return {"events": [f"{name} done"]}

    return node


builder = StateGraph(state_schema=FanoutState)
for name in ["a", "b", "c"]:
    builder.add_node(name, make_node(name))

builder.add_conditional_edges(
    START,
    route,
    path_map={"a": "a", "b": "b", "c": "c"},
)
for name in ["a", "b", "c"]:
    builder.add_edge(name, END)

graph = builder.compile()
for selected in [["a"], ["a", "b"], ["a", "b", "c"]]:
    result = graph.invoke({"selected": selected, "events": []})
    print(selected, "->", sorted(result["events"]))
# ['a'] -> ['a done']
# ['a', 'b'] -> ['a done', 'b done']
# ['a', 'b', 'c'] -> ['a done', 'b done', 'c done']

> **汇合陷阱**：如果一个节点同时接收多条普通边，不应想当然地认为它只执行一次。需要“所有并行分支完成后只执行一次”时，将汇合节点声明为 `defer=True`。上一案例可将 `builder.add_node("review", review)` 改为下面的写法。

In [ ]:
# 只展示需要替换的构建部分：defer 让 review 在并行工作完成后执行一次

builder = StateGraph(state_schema=OrderState)
for name, node in {
    "classify": classify,
    "normal_prepare": normal_prepare,
    "priority_prepare": priority_prepare,
    "package": package,
    "notify": notify,
    "review": review,
    "restock": restock,
    "finish": finish,
    "fail": fail,
}.items():
    if name == "review":
        builder.add_node(name, node, defer=True)
    else:
        builder.add_node(name, node)

builder.add_edge(START, "classify")
builder.add_edge("normal_prepare", "package")
builder.add_edge("priority_prepare", "package")
builder.add_edge("package", "review")
builder.add_edge("package", "notify")
builder.add_edge("notify", "review")
builder.add_conditional_edges("review", route_after_review, {
    "restock": "restock",
    "done": "finish",
    "failed": "fail",
})
builder.add_edge("restock", "review")
builder.add_edge("finish", END)
builder.add_edge("fail", END)

result = builder.compile().invoke({
    "order_type": "priority", "stock": 0, "need": 2,
    "retries": 0, "events": [],
})
print(result["retries"], result["events"][-3:])
# 1 ['复核:库存充足', '订单完成']（并行汇合后的 review 只执行一次）

# 6. 本章小结

## 6.1 控制流选型决策表

| 需求 | 用什么 | 关键代码 |
|---|---|---|
| 下游固定，全部并行执行 | 静态分支 | `add_edge("a", "b"); add_edge("a", "c")` |
| 下游按 State 从固定候选中选 | 条件分支 | `add_conditional_edges("a", route, path_map)` |
| 候选固定但一次可选多个 | 条件分支多目标 | 路由函数返回 `list[str]` |
| 下游数量/输入运行时才知道 | 动态分支 | Node 返回 `Command(goto=[Send(...)])` |
| 并行结果汇合后再处理 | Defer 节点 + reducer | `add_node("agg", agg, defer=True)` |
| 反复执行直到满足条件 | 循环 | 条件边 / `Command(goto=...)` 指回上游 |
| 节点内边更新 State 边决定走向 | Command | `return Command(goto=..., update=...)` |
| 观察/调试执行路径 | stream | `graph.stream(input, stream_mode="updates")` |

## 6.2 常见坑清单（全部在本环境实测过）

1. **并行节点写同一个无 reducer 的 key** → `InvalidUpdateError: Can receive only one value per step`。修复：加 reducer，或只让一个节点写
2. **路由函数返回 `Command`** → 不报错但**被静默忽略**（跳转 + 更新都丢失）。Command 只能由 Node 返回
3. **普通边与条件边/Command 共存** → 叠加语义：两条都走；与 goto 指向同一节点时去重。每个节点只用一种出边方式
4. **路由返回值不在映射表** → 运行时 `KeyError`（编译期不检查），所有分支路径都要测到
5. **并行节点读同一快照** → 同 super-step 的节点互相看不到对方的更新；要汇合用 defer 或 reducer
6. **循环没有出口** → `GraphRecursionError`（默认 25 个 super-step）。长循环要显式设 `recursion_limit`，并在 Node 里做业务上限

## 6.3 后续预告

- **持久化与人工介入（interrupt）**：第 1 章提过「人工介入」是 LangGraph 核心能力——在关键 Node 暂停等用户确认，配合 checkpointer（第 2 章末尾已铺垫 `MemorySaver`）实现断点续跑，适合单独成章展开
- **子图（Subgraph）**：把编译好的 Graph 作为 Node 组合复用，是构建多 agent 系统的基础